# Japanese Used-Car Fair Price Estimator — Production Model Training

## Goal

Train the model that powers the production app:

> **User enters vehicle details → app returns estimated fair listing price + expected market range.**

This notebook follows the decisions established in the EDA notebook.

### Outputs

The notebook trains and exports:

1. a **point-estimate CatBoost model** for fair price,
2. a **held-out conformal calibration value** used to create the price range,
3. model metadata / feature contract,
4. UI category options for common dropdowns.

The point model is trained on `log1p(vehicle_price_jpy)` to reduce the influence of extremely expensive vehicles. Predictions are converted back to JPY for all business metrics.

The displayed range is calibrated on data that is not used to fit the model. This avoids training two additional interval models and gives us a direct test-set coverage measurement.

### Production design principle

The feature engineering in this notebook is written as reusable Python functions.  
The same functions should be used by the API/backend so training and inference do not implement different transformations.

## 1. Dependencies

Install the packages once in your project environment if necessary:

```bash
pip install pandas numpy scikit-learn catboost matplotlib joblib
```

CatBoost is selected because it handles:

- nonlinear relationships,
- high-cardinality categorical features,
- unseen categorical values through hashing,
- missing numerical values,
- interactions such as model × year × mileage.

Run from the project root or the `notebook` folder in the project virtual environment. Dependencies are installed from `requirements.txt`; LightGBM is also required for this comparison.


In [ ]:
from pathlib import Path
import json
import math
import re
import unicodedata
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42

# Target sanity bounds found during EDA.
MIN_TARGET_JPY = 30_000
MAX_TARGET_JPY = 100_000_000

LOCAL_PATH = Path("../data/raw/japan-cars-csv_new.csv") if Path.cwd().name == "notebook" else Path("data/raw/japan-cars-csv_new.csv")

if LOCAL_PATH.exists():
    DATA_PATH = LOCAL_PATH
else:
    raise FileNotFoundError(
        "Place 'japan-cars-csv_new.csv' next to this notebook or update DATA_PATH."
    )

ARTIFACT_DIR = Path("artifacts") if Path.cwd().name == "notebook" else Path("notebook/artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Dataset:   {DATA_PATH.resolve()}")
print(f"Artifacts: {ARTIFACT_DIR.resolve()}")

## 2. Load and define the modeling population

We start from scraper rows marked `ready`, then apply explicit target sanity limits.

We do not train on `review` rows because their core fields may be absent.  
We also do not silently remove arbitrary statistical outliers: only the documented target bounds are applied here.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

model_df = df.loc[
    (df["quality_status"] == "ready")
    & df["vehicle_price_jpy"].between(MIN_TARGET_JPY, MAX_TARGET_JPY)
].copy()

print(f"Raw rows:      {len(df):,}")
print(f"Training pool: {len(model_df):,}")
print(f"Target median: ¥{model_df['vehicle_price_jpy'].median():,.0f}")

## 3. Define the production input contract

These are fields that the frontend may send to the prediction API.

### Mandatory

- manufacturer
- model
- registration year
- mileage

### Recommended but allowed to be unknown

- trim
- fuel
- transmission
- drivetrain
- repair history
- prefecture

The model intentionally excludes source website, dealer name, listing title, fees, and total price.

This keeps the model aligned with the real user workflow rather than optimizing only for an offline benchmark.

In [ ]:
RAW_INPUT_FEATURES = [
    "manufacturer",
    "model",
    "trim",
    "registration_year",
    "mileage_km",
    "fuel",
    "transmission",
    "drivetrain",
    "repair_history",
    "prefecture",
]

MANDATORY_INPUTS = [
    "manufacturer",
    "model",
    "registration_year",
    "mileage_km",
]

OPTIONAL_INPUTS = sorted(set(RAW_INPUT_FEATURES) - set(MANDATORY_INPUTS))

print("Mandatory UI/API inputs:", MANDATORY_INPUTS)
print("Optional/recommended inputs:", OPTIONAL_INPUTS)

## 4. Reusable preprocessing and feature engineering

### Why this section matters

In production, preprocessing bugs can be more damaging than model-choice differences.  
For example, if training maps `軽油` to diesel but the API leaves it untouched, the model receives a different category distribution.

The functions below should eventually live in a shared module such as:

`src/features.py`

and be imported by both training code and the prediction API.

### Japanese handling

We preserve manufacturer/model/trim text. We only normalize Unicode, whitespace, placeholders, and selected semantic synonyms.

In [ ]:
def normalize_text(value):
    """Return a stable string token for scraped/user categorical values."""
    if pd.isna(value):
        return "__UNKNOWN__"

    text = unicodedata.normalize("NFKC", str(value)).strip()
    text = re.sub(r"\s+", " ", text)

    placeholders = {"", "-", "−", "'-", "不明", "nan", "None"}
    return "__UNKNOWN__" if text in placeholders else text


def normalize_fuel(value):
    """Collapse fuel synonyms into production-safe canonical values."""
    text = normalize_text(value)

    if text == "__UNKNOWN__":
        return text
    if "プラグイン" in text and "ハイブリッド" in text:
        return "phev"
    if "電気" in text or "EV" in text.upper():
        return "electric"
    if "ディーゼル" in text or "軽油" in text:
        return "diesel"
    if "ハイブリッド" in text:
        return "hybrid"
    if "ガソリン" in text:
        return "gasoline"
    if "水素" in text:
        return "hydrogen"
    return "other"


def normalize_transmission(value):
    """Convert detailed transmission strings into a stable family."""
    text = normalize_text(value).upper()

    if text == "__UNKNOWN__":
        return text
    if "CVT" in text:
        return "CVT"
    if "DCT" in text:
        return "DCT"
    # MT mode on an AT/CVT is not the same as a manual gearbox.
    if "AT" in text:
        return "AT"
    if "MT" in text:
        return "MT"
    return "other"


def extract_gear_count(value):
    """Extract a stated gear count such as 6 from 6AT, 6MT, or 6速."""
    text = normalize_text(value).upper()

    if text == "__UNKNOWN__":
        return np.nan

    match = re.search(r"(\d+)\s*(?:AT|MT|速)", text)
    return float(match.group(1)) if match else np.nan


def normalize_repair_history(value):
    """Preserve a true three-state concept: yes / no / unknown."""
    if pd.isna(value):
        return "__UNKNOWN__"

    if isinstance(value, (bool, np.bool_)):
        return "yes" if value else "no"

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return "yes"
    if text in {"false", "0", "no"}:
        return "no"

    return "__UNKNOWN__"


MANUFACTURER_CANONICAL_MAP = {
    "AMG": "メルセデスAMG",
    "メルセデスAMG": "メルセデスAMG",
    "メルセデス・マイバッハ": "メルセデス・マイバッハ",
    "メルセデスマイバッハ": "メルセデス・マイバッハ",
    "MINI": "MINI",
    "ミニ": "MINI",
}

MANUFACTURER_GROUP_MAP = {
    "メルセデスAMG": "メルセデス・ベンツ",
    "メルセデス・マイバッハ": "メルセデス・ベンツ",
}


def canonicalize_manufacturer(value):
    """Merge spelling/source aliases while preserving AMG as a marque."""
    normalized = normalize_text(value)
    return MANUFACTURER_CANONICAL_MAP.get(normalized, normalized)


def manufacturer_group(value):
    """Return a corporate parent without replacing the canonical marque."""
    canonical = canonicalize_manufacturer(value)
    return MANUFACTURER_GROUP_MAP.get(canonical, canonical)


# The reference year is the market vintage represented by this model.
# It is deliberately derived from training data and exported with the model.
collection_years = pd.to_datetime(
    model_df["collected_at"], errors="coerce", utc=True
).dt.year.dropna()

MODEL_REFERENCE_YEAR = int(collection_years.mode().iloc[0])

print("Model reference year:", MODEL_REFERENCE_YEAR)

In [ ]:
MODEL_FEATURES = [
    "manufacturer",
    "manufacturer_group",
    "model",
    "trim",
    "registration_year",
    "mileage_km",
    "vehicle_age",
    "log_mileage",
    "km_per_year",
    "fuel_clean",
    "transmission_simple",
    "gear_count",
    "manual_mode",
    "drivetrain",
    "repair_history_clean",
    "prefecture",
]

CATEGORICAL_FEATURES = [
    "manufacturer",
    "manufacturer_group",
    "model",
    "trim",
    "fuel_clean",
    "transmission_simple",
    "drivetrain",
    "repair_history_clean",
    "prefecture",
]


def build_model_features(raw_frame, reference_year=MODEL_REFERENCE_YEAR):
    """
    Convert raw UI/scraper columns into exactly the features expected by the model.

    Parameters
    ----------
    raw_frame : pandas.DataFrame
        Must contain RAW_INPUT_FEATURES. Optional values may be null.
    reference_year : int
        Market year represented by the current trained model.

    Returns
    -------
    pandas.DataFrame
        Model-ready features in a fixed column order.
    """
    x = raw_frame.copy()

    # Guarantee that optional fields exist when this function is called
    # from an API payload that may omit them.
    for col in RAW_INPUT_FEATURES:
        if col not in x.columns:
            x[col] = np.nan

    # Clean identity/location categorical text without translating it.
    x["manufacturer_group"] = x["manufacturer"].map(manufacturer_group)
    x["manufacturer"] = x["manufacturer"].map(canonicalize_manufacturer)
    for col in ["model", "trim", "drivetrain", "prefecture"]:
        x[col] = x[col].map(normalize_text)

    # Canonical derived categoricals.
    x["fuel_clean"] = x["fuel"].map(normalize_fuel)
    x["transmission_simple"] = x["transmission"].map(normalize_transmission)
    x["repair_history_clean"] = x["repair_history"].map(
        normalize_repair_history
    )

    # Transmission-derived numerical indicators.
    x["gear_count"] = x["transmission"].map(extract_gear_count)
    x["manual_mode"] = x["transmission"].map(
        lambda value: int("MTモード" in normalize_text(value))
    )

    # Coerce user/scraped numerical inputs safely.
    x["registration_year"] = pd.to_numeric(
        x["registration_year"], errors="coerce"
    )
    x["mileage_km"] = pd.to_numeric(x["mileage_km"], errors="coerce")

    # Derived features.
    x["vehicle_age"] = (
        reference_year - x["registration_year"]
    ).clip(lower=0)

    x["log_mileage"] = np.log1p(x["mileage_km"].clip(lower=0))

    # Use at least one year in the denominator so nearly-new vehicles
    # do not create an infinite or undefined rate.
    x["km_per_year"] = (
        x["mileage_km"] / x["vehicle_age"].clip(lower=1)
    )

    return x[MODEL_FEATURES]

## 5. Build X/y and run integrity checks

The model is trained only on the production-compatible feature matrix.

Before fitting, we verify:

- mandatory fields are available,
- target is finite and positive,
- model columns are in the exact expected order,
- categorical features contain string tokens rather than nulls.

In [ ]:
X = build_model_features(model_df[RAW_INPUT_FEATURES])
y = model_df["vehicle_price_jpy"].astype(float)

mandatory_missing = model_df[MANDATORY_INPUTS].isna().sum()
print("Missing mandatory inputs inside training pool:")
display(mandatory_missing.to_frame("missing_rows"))

assert np.isfinite(y).all(), "Target contains non-finite values."
assert (y > 0).all(), "Target must be positive."
assert X.columns.tolist() == MODEL_FEATURES
assert X[CATEGORICAL_FEATURES].isna().sum().sum() == 0

print(f"Feature matrix shape: {X.shape}")
display(X.head())

## 6. Leakage-resistant train / validation / calibration / test split

Rows sharing the same canonical manufacturer, model, trim, registration year, mileage, and target price are kept in one partition. This prevents repeated inventory—especially vehicles appearing on both marketplaces—from making holdout performance look better than it is.

The approximate allocation remains 70% train, 10% validation, 10% calibration, and 10% test. Distribution tables verify that source and price composition remain similar after grouped splitting.


In [ ]:
# A repeated listing group must never cross evaluation boundaries.
group_key = X[[
    "manufacturer", "model", "trim", "registration_year", "mileage_km"
]].copy()
group_key["vehicle_price_jpy"] = y
vehicle_group = pd.util.hash_pandas_object(group_key, index=False).astype(str)


def grouped_partition(frame, target, groups, test_size, random_state):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state,
    )
    left_pos, right_pos = next(splitter.split(frame, target, groups=groups))
    return (
        frame.iloc[left_pos], frame.iloc[right_pos],
        target.iloc[left_pos], target.iloc[right_pos],
        groups.iloc[left_pos], groups.iloc[right_pos],
    )


X_train, X_temp, y_train, y_temp, groups_train, groups_temp = grouped_partition(
    X, y, vehicle_group, test_size=0.30, random_state=RANDOM_STATE
)
X_valid, X_hold, y_valid, y_hold, groups_valid, groups_hold = grouped_partition(
    X_temp, y_temp, groups_temp, test_size=2/3, random_state=RANDOM_STATE + 1
)
X_calib, X_test, y_calib, y_test, groups_calib, groups_test = grouped_partition(
    X_hold, y_hold, groups_hold, test_size=0.50, random_state=RANDOM_STATE + 2
)

idx_train, idx_valid = X_train.index, X_valid.index
idx_calib, idx_test = X_calib.index, X_test.index

partition_groups = {
    "train": set(groups_train),
    "validation": set(groups_valid),
    "calibration": set(groups_calib),
    "test": set(groups_test),
}
partition_names = list(partition_groups)
for i, left in enumerate(partition_names):
    for right in partition_names[i + 1:]:
        assert partition_groups[left].isdisjoint(partition_groups[right]), (
            f"Repeated vehicle group leaked between {left} and {right}."
        )

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_valid), len(X_calib), len(X_test)],
    "unique_vehicle_groups": [
        groups_train.nunique(), groups_valid.nunique(),
        groups_calib.nunique(), groups_test.nunique(),
    ],
    "median_price_jpy": [
        y_train.median(), y_valid.median(), y_calib.median(), y_test.median(),
    ],
}, index=["train", "validation", "calibration", "test"])
display(split_summary)

source_mix = pd.DataFrame({
    "train": model_df.loc[idx_train, "source"].value_counts(normalize=True),
    "validation": model_df.loc[idx_valid, "source"].value_counts(normalize=True),
    "calibration": model_df.loc[idx_calib, "source"].value_counts(normalize=True),
    "test": model_df.loc[idx_test, "source"].value_counts(normalize=True),
}).fillna(0)
display(source_mix)


## 7. Evaluation functions

We evaluate in real Japanese yen, even though the point model is trained on log-price.

Metrics:

- **MAE:** average absolute JPY error.
- **Median AE:** typical absolute error, less sensitive to a few exotic cars.
- **RMSE:** penalizes large misses more strongly.
- **R²:** variance explained, useful but not sufficient by itself.
- **Within ±10% / ±20%:** highly interpretable product metrics.
- **sMAPE:** bounded relative error metric that behaves better than ordinary MAPE around low prices.

In [ ]:
def regression_metrics(y_true, y_pred):
    """Return business-friendly regression metrics."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    abs_error = np.abs(y_true - y_pred)
    pct_error = abs_error / np.maximum(y_true, 1.0)

    smape = np.mean(
        2 * abs_error / np.maximum(np.abs(y_true) + np.abs(y_pred), 1.0)
    )

    return {
        "MAE_JPY": mean_absolute_error(y_true, y_pred),
        "Median_AE_JPY": median_absolute_error(y_true, y_pred),
        "RMSE_JPY": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "sMAPE": smape,
        "Within_10pct": np.mean(pct_error <= 0.10),
        "Within_20pct": np.mean(pct_error <= 0.20),
    }


def format_metrics(metrics):
    """Pretty-print the dictionary while retaining raw values elsewhere."""
    formatted = {}
    for key, value in metrics.items():
        if key.endswith("_JPY"):
            formatted[key] = f"¥{value:,.0f}"
        elif key in {"R2", "sMAPE", "Within_10pct", "Within_20pct"}:
            formatted[key] = f"{value:.2%}" if key != "R2" else f"{value:.3f}"
        else:
            formatted[key] = value
    return formatted

## 8. Business baseline

A professional model should beat a simple rule.

The baseline predicts the training median for the most specific available grouping:

1. manufacturer + model + registration year,
2. manufacturer + model,
3. manufacturer,
4. global median.

This is a surprisingly strong baseline for used-car valuation and makes model improvement meaningful.

In [ ]:
train_keys = X.loc[
    idx_train,
    ["manufacturer", "model", "registration_year"],
].copy()
train_keys["target"] = y_train.values

test_keys = X.loc[
    idx_test,
    ["manufacturer", "model", "registration_year"],
].copy()

median_model_year = train_keys.groupby(
    ["manufacturer", "model", "registration_year"]
)["target"].median()

median_model = train_keys.groupby(
    ["manufacturer", "model"]
)["target"].median()

median_manufacturer = train_keys.groupby(
    "manufacturer"
)["target"].median()

global_median = float(y_train.median())


def predict_baseline(row):
    model_year_key = (
        row["manufacturer"],
        row["model"],
        row["registration_year"],
    )
    if model_year_key in median_model_year.index:
        return median_model_year.loc[model_year_key]

    model_key = (row["manufacturer"], row["model"])
    if model_key in median_model.index:
        return median_model.loc[model_key]

    if row["manufacturer"] in median_manufacturer.index:
        return median_manufacturer.loc[row["manufacturer"]]

    return global_median


baseline_pred = test_keys.apply(predict_baseline, axis=1).to_numpy()
baseline_metrics = regression_metrics(y_test, baseline_pred)

display(pd.DataFrame([format_metrics(baseline_metrics)], index=["Median baseline"]))

## 9. Train candidate models

We now compare three strong gradient-boosting algorithms:

- CatBoost
- LightGBM
- XGBoost

All models use the exact same train/validation/test rows and predict
`log1p(vehicle_price_jpy)`.

CatBoost can directly process categorical strings.

LightGBM and XGBoost require numerical representations, so their pipelines
use an OrdinalEncoder fitted only on the training data.

The test set is NOT used for model selection.

In [ ]:
#5. Add a numeric-feature list

NUMERIC_FEATURES = [
    col
    for col in MODEL_FEATURES
    if col not in CATEGORICAL_FEATURES
]

print("Categorical features:")
print(CATEGORICAL_FEATURES)

print("\nNumeric features:")
print(NUMERIC_FEATURES)

In [ ]:
#Create preprocessing for LightGBM and XGBoost . CatBoost doesn't need this. But LightGBM and XGBoost cannot directly use strings
def create_tree_preprocessor():
    """
    Create preprocessing for models that require numerical inputs.

    Categorical values:
        Missing -> most frequent
        Strings -> integer category codes

    Numerical values:
        Missing -> median
    """

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                )
            ),
        ]
    )

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            )
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_FEATURES,
            ),
            (
                "numerical",
                numeric_pipeline,
                NUMERIC_FEATURES,
            ),
        ]
    )

In [ ]:
#create the cordinate for the models
def create_candidate_models():

    catboost_model = CatBoostRegressor(
        iterations=900,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=8,
        random_strength=0.5,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        allow_writing_files=False,
        verbose=False,
    )


    lightgbm_model = Pipeline(
        steps=[
            (
                "preprocessor",
                create_tree_preprocessor()
            ),
            (
                "model",
                LGBMRegressor(
                    n_estimators=900,
                    learning_rate=0.05,
                    num_leaves=63,
                    max_depth=-1,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    random_state=RANDOM_STATE,
                    n_jobs=4,
                )
            ),
        ]
    )


    xgboost_model = Pipeline(
        steps=[
            (
                "preprocessor",
                create_tree_preprocessor()
            ),
            (
                "model",
                XGBRegressor(
                    n_estimators=900,
                    learning_rate=0.05,
                    max_depth=7,
                    min_child_weight=3,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="reg:squarederror",
                    random_state=RANDOM_STATE,
                    n_jobs=4,
                    tree_method="hist",
                )
            ),
        ]
    )


    return {
        "CatBoost": catboost_model,
        "LightGBM": lightgbm_model,
        "XGBoost": xgboost_model,
    }

In [ ]:
#prediction function for all models
def predict_price_jpy(
    model,
    X,
):
    """
    Models predict log1p(price).

    Convert the prediction back to Japanese yen.
    """

    predicted_log_price = model.predict(X)

    predicted_price = np.expm1(
        predicted_log_price
    )

    # Price should never be negative.
    predicted_price = np.clip(
        predicted_price,
        0,
        None,
    )

    return predicted_price

In [ ]:
def fit_candidate_model(
    model_name,
    model,
    X_train,
    y_train,
):
    """
    Train every candidate on log1p(price).

    Using the same target transformation makes
    the model comparison more fair.
    """

    y_train_log = np.log1p(y_train)

    if model_name == "CatBoost":

        model.fit(
            X_train,
            y_train_log,
            cat_features=CATEGORICAL_FEATURES,
        )

    else:

        model.fit(
            X_train,
            y_train_log,
        )

    return model
#train all models 
candidate_models = create_candidate_models()

trained_models = {}

for model_name, model in candidate_models.items():

    print("\n")
    print("=" * 70)
    print(f"Training: {model_name}")
    print("=" * 70)

    trained_model = fit_candidate_model(
        model_name=model_name,
        model=model,
        X_train=X_train,
        y_train=y_train,
    )

    trained_models[
        model_name
    ] = trained_model

## 10. Validation-set model comparison

The test set has not been used for training or early stopping.

We compare the CatBoost model with the business baseline using the same rows.

In [ ]:
validation_results = []


for model_name, model in trained_models.items():

    predictions = predict_price_jpy(
        model,
        X_valid,
    )

    metrics = regression_metrics(
        y_valid,
        predictions,
    )

    validation_results.append(
        {
            "model": model_name,
            **metrics,
        }
    )


validation_leaderboard = (
    pd.DataFrame(validation_results)
    .sort_values(
        "MAE_JPY",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(validation_leaderboard)

In [ ]:
# Automatically identify the winner


BEST_MODEL_NAME = (
    validation_leaderboard
    .iloc[0]["model"]
)

print(
    f"Best validation model: "
    f"{BEST_MODEL_NAME}"
)

In [ ]:
#Compare against your baseline too

baseline_valid_keys = X.loc[
    y_valid.index,
    [
        "manufacturer",
        "model",
        "registration_year",
    ],
].copy()


baseline_valid_pred = (
    baseline_valid_keys
    .apply(
        predict_baseline,
        axis=1
    )
    .to_numpy()
)


baseline_valid_metrics = regression_metrics(
    y_valid,
    baseline_valid_pred,
)


baseline_row = pd.DataFrame(
    [
        {
            "model": "Median baseline",
            **baseline_valid_metrics,
        }
    ]
)


comparison_table = pd.concat(
    [
        baseline_row,
        validation_leaderboard,
    ],
    ignore_index=True,
).sort_values(
    "MAE_JPY"
)


display(comparison_table)

## 11. Refit selected model and perform final test



In [ ]:
X_train_final = pd.concat(
    [
        X_train,
        X_valid,
    ],
    axis=0,
)

y_train_final = pd.concat(
    [
        y_train,
        y_valid,
    ],
    axis=0,
)


print(
    "Final training rows:",
    len(X_train_final)
)

In [ ]:
final_candidates = (
    create_candidate_models()
)

final_model = final_candidates[
    BEST_MODEL_NAME
]

In [ ]:
final_model = fit_candidate_model(
    model_name=BEST_MODEL_NAME,
    model=final_model,
    X_train=X_train_final,
    y_train=y_train_final,
)

In [ ]:
test_pred = predict_price_jpy(
    final_model,
    X_test,
)


test_metrics = regression_metrics(
    y_test,
    test_pred,
)


display(
    pd.DataFrame(
        [test_metrics],
        index=[
            BEST_MODEL_NAME
        ]
    )
)

In [ ]:

results = pd.DataFrame({
    "actual_jpy": y_test.to_numpy(),
    "predicted_jpy": test_pred,
}, index=y_test.index)

results["residual_jpy"] = (
    results["actual_jpy"] - results["predicted_jpy"]
)

results["abs_error_jpy"] = np.abs(results["residual_jpy"])

results["abs_pct_error"] = (
    results["abs_error_jpy"] / results["actual_jpy"]
)

display(results.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T)

In [ ]:
# Use a central range for readable plots while keeping all rows in metrics.
plot_limit = results["actual_jpy"].quantile(0.99)
plot_results = results[
    (results["actual_jpy"] <= plot_limit)
    & (results["predicted_jpy"] <= plot_limit)
].copy()

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(
    plot_results["actual_jpy"] / 1_000_000,
    plot_results["predicted_jpy"] / 1_000_000,
    s=10,
    alpha=0.25,
)
max_m = max(
    plot_results["actual_jpy"].max(),
    plot_results["predicted_jpy"].max(),
) / 1_000_000
ax.plot([0, max_m], [0, max_m], linestyle="--")
ax.set_title("Actual vs predicted price")
ax.set_xlabel("Actual price (million JPY)")
ax.set_ylabel("Predicted price (million JPY)")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    plot_results["predicted_jpy"] / 1_000_000,
    plot_results["residual_jpy"] / 1_000_000,
    s=10,
    alpha=0.25,
)
ax.axhline(0, linestyle="--")
ax.set_title("Residuals vs predicted price")
ax.set_xlabel("Predicted price (million JPY)")
ax.set_ylabel("Actual - predicted (million JPY)")
plt.show()

## 12. Segment-level evaluation

Production quality should be checked by important slices, not only globally.

We report performance by:

- source marketplace,
- top manufacturers,
- price bands,
- vehicle-age bands.

The source is used here only for diagnostics; it is not a model input.

In [ ]:
test_context = model_df.loc[
    y_test.index,
    [
        "source",
        "manufacturer",
        "registration_year",
        "mileage_km",
    ],
].copy()

test_context["manufacturer"] = X.loc[y_test.index, "manufacturer"]

test_context["actual_jpy"] = y_test
test_context["predicted_jpy"] = test_pred
test_context["vehicle_age"] = (
    MODEL_REFERENCE_YEAR - test_context["registration_year"]
).clip(lower=0)


def segment_metrics(frame, segment_col, min_rows=50):
    rows = []

    for segment, group in frame.groupby(segment_col, dropna=False):
        if len(group) < min_rows:
            continue

        metrics = regression_metrics(
            group["actual_jpy"],
            group["predicted_jpy"],
        )

        rows.append({
            segment_col: segment,
            "rows": len(group),
            **metrics,
        })

    return (
        pd.DataFrame(rows)
        .sort_values("rows", ascending=False)
        .reset_index(drop=True)
    )


print("By source:")
display(segment_metrics(test_context, "source", min_rows=20))

top_manufacturers = (
    test_context["manufacturer"]
    .value_counts()
    .head(15)
    .index
)
manufacturer_eval = test_context[
    test_context["manufacturer"].isin(top_manufacturers)
]
print("\nTop manufacturers:")
display(segment_metrics(manufacturer_eval, "manufacturer", min_rows=50))

In [ ]:
test_context["price_band"] = pd.cut(
    test_context["actual_jpy"],
    bins=[0, 500_000, 1_000_000, 2_000_000, 3_500_000, 5_000_000, 10_000_000, np.inf],
    labels=[
        "<0.5M",
        "0.5-1M",
        "1-2M",
        "2-3.5M",
        "3.5-5M",
        "5-10M",
        "10M+",
    ],
)

test_context["age_band"] = pd.cut(
    test_context["vehicle_age"],
    bins=[-1, 2, 5, 10, 15, 20, np.inf],
    labels=["0-2", "3-5", "6-10", "11-15", "16-20", "21+"],
)

print("By actual price band:")
display(segment_metrics(test_context, "price_band", min_rows=30))

print("\nBy vehicle-age band:")
display(segment_metrics(test_context, "age_band", min_rows=30))

## 15. Calibrate the expected price range

A production valuation app should not imply false precision.

Instead of training two additional quantile models, we use a **split-conformal-style interval on log price**:

1. train the main model on the training set,
2. choose boosting iterations on the validation set,
3. predict the untouched calibration set,
4. measure absolute errors in log-price space,
5. choose an error quantile corresponding to the desired coverage,
6. apply that width around future log-price predictions.

Because the model is trained in log space, the final interval is naturally multiplicative: expensive vehicles receive wider JPY ranges than inexpensive ones.

We target **80% coverage** here. This is a practical product choice, not a universal requirement. A wider target such as 90% produces wider intervals.

Conformal coverage relies on the calibration and future examples being sufficiently exchangeable. Market drift can break that assumption, so coverage must be monitored over time.

In [ ]:
# Desired probability that the displayed range contains the actual listing price.
TARGET_INTERVAL_COVERAGE = 0.80
alpha = 1.0 - TARGET_INTERVAL_COVERAGE

# Work in the same log-price space used to train the point model.
calib_pred_log = final_model.predict(X_calib)
calib_true_log = np.log1p(y_calib.to_numpy())

# Nonconformity score: absolute log-price prediction error.
calibration_scores = np.abs(calib_true_log - calib_pred_log)

# Finite-sample conformal quantile.
# The ceil((n + 1) * (1 - alpha)) / n correction avoids using
# a quantile that is too optimistic for a finite calibration sample.
n_calib = len(calibration_scores)
quantile_level = min(
    1.0,
    math.ceil((n_calib + 1) * (1.0 - alpha)) / n_calib,
)

try:
    CONFORMAL_Q_HAT = float(
        np.quantile(
            calibration_scores,
            quantile_level,
            method="higher",
        )
    )
except TypeError:
    # Compatibility with older NumPy versions.
    CONFORMAL_Q_HAT = float(
        np.quantile(
            calibration_scores,
            quantile_level,
            interpolation="higher",
        )
    )

print(f"Calibration rows: {n_calib:,}")
print(f"Target interval coverage: {TARGET_INTERVAL_COVERAGE:.0%}")
print(f"Conformal log-error width q_hat: {CONFORMAL_Q_HAT:.4f}")

In [ ]:
# Evaluate interval coverage only on the untouched test set.
test_pred_log = final_model.predict(X_test)

interval_low = np.expm1(test_pred_log - CONFORMAL_Q_HAT)
interval_high = np.expm1(test_pred_log + CONFORMAL_Q_HAT)

interval_low = np.clip(interval_low, 0, None)
interval_high = np.clip(interval_high, 0, None)

coverage = np.mean(
    (y_test.to_numpy() >= interval_low)
    & (y_test.to_numpy() <= interval_high)
)

interval_width = interval_high - interval_low

print(f"Observed test interval coverage: {coverage:.2%}")
print(f"Median interval width: ¥{np.median(interval_width):,.0f}")
print(f"Mean interval width:   ¥{np.mean(interval_width):,.0f}")

interval_eval = pd.DataFrame({
    "actual_jpy": y_test.to_numpy(),
    "point_jpy": test_pred,
    "low_jpy": interval_low,
    "high_jpy": interval_high,
})

display(interval_eval.head(10))

## 16. Production inference function

This is the shape of the function our FastAPI backend can call.

It performs four steps:

1. validate mandatory user inputs,
2. run the exact training-time feature engineering,
3. generate the point prediction in log-price space,
4. apply the stored conformal calibration width and return JSON-ready JPY values.

Unknown optional fields are accepted.

The deployed API therefore needs only:

- the CatBoost point model,
- the preprocessing code,
- the exported conformal `q_hat`,
- the feature metadata.

In [ ]:
def validate_user_input(payload):
    """Validate mandatory fields and basic numerical plausibility."""
    missing = [
        field
        for field in MANDATORY_INPUTS
        if payload.get(field) is None or payload.get(field) == ""
    ]

    if missing:
        raise ValueError(f"Missing mandatory input(s): {missing}")

    registration_year = float(payload["registration_year"])
    mileage_km = float(payload["mileage_km"])

    if not (1900 <= registration_year <= MODEL_REFERENCE_YEAR + 1):
        raise ValueError("registration_year is outside the supported range.")

    if mileage_km < 0:
        raise ValueError("mileage_km cannot be negative.")


def predict_fair_price(payload):
    """
    Predict fair listing-market price and calibrated range for one vehicle.

    Returns integer JPY values suitable for an API response.
    """
    validate_user_input(payload)

    raw = pd.DataFrame([payload])
    features = build_model_features(
        raw,
        reference_year=MODEL_REFERENCE_YEAR,
    )

    # The point model predicts log1p(price).
    point_log = float(final_model.predict(features)[0])
    
    # Convert the point and conformal bounds back into JPY.
    point = float(np.expm1(point_log))
    low = float(np.expm1(point_log - CONFORMAL_Q_HAT))
    high = float(np.expm1(point_log + CONFORMAL_Q_HAT))

    return {
        "estimated_fair_price_jpy": int(round(max(point, 0))),
        "expected_range_low_jpy": int(round(max(low, 0))),
        "expected_range_high_jpy": int(round(max(high, 0))),
        "target_interval_coverage": TARGET_INTERVAL_COVERAGE,
        "model_reference_year": MODEL_REFERENCE_YEAR,
        "valuation_basis": "advertised_vehicle_price",
    }

## 17. Example app request

The API can accept Japanese or English/canonical values depending on your frontend design, provided the backend preprocessing remains consistent.

A realistic payload might come from a form such as:

- Toyota
- Alphard
- 2021
- 42,000 km
- gasoline
- automatic/CVT
- 2WD
- no repair history
- Tokyo

The example below uses values that resemble the Japanese source data.

In [ ]:
example_user_input = {
    "manufacturer": "トヨタ",
    "model": "アルファード",
    "trim": "2.5 S Cパッケージ",
    "registration_year": 2021,
    "mileage_km": 42_000,
    "fuel": "ガソリン",
    "transmission": "インパネCVT",
    "drivetrain": "2WD",
    "repair_history": False,
    "prefecture": "東京都",
}

example_prediction = predict_fair_price(example_user_input)
display(example_prediction)

## 18. Build UI dropdown options from training data

The frontend should ideally use controlled choices rather than unrestricted free text for make/model.

The exported JSON below provides common values from the training snapshot.  
In a full application you may instead store this hierarchy in a database and filter model choices after the user selects a manufacturer.

In [ ]:
# Build dependent frontend choices from combinations observed in the training population.
# This prevents impossible choices such as EV for a model/year with no EV listings.
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ui_options import build_ui_options

ui_options = build_ui_options(df)

UI_OPTIONS_PATH = ARTIFACT_DIR / "ui_options.json"
with open(UI_OPTIONS_PATH, "w", encoding="utf-8") as file:
    json.dump(ui_options, file, ensure_ascii=False, separators=(",", ":"))

configuration_count = sum(
    len(years)
    for models in ui_options["vehicle_options"].values()
    for years in models.values()
)
print("Saved:", UI_OPTIONS_PATH)
print("Canonical manufacturers:", len(ui_options["manufacturer_to_models"]))
print("Make/model/year combinations:", configuration_count)


## 19. Export the production artifacts

The exported metadata is just as important as the model file.

It records:

- feature order,
- categorical fields,
- raw API contract,
- reference year,
- target definition,
- price bounds,
- dataset collection period,
- point-model metrics,
- conformal interval calibration.

This helps prevent a future API from loading the correct `.cbm` file but feeding it the wrong fields or applying the wrong uncertainty range.

In [ ]:
import joblib
MODEL_PATH = (
    ARTIFACT_DIR
    / "fair_price_model.joblib"
)


joblib.dump(
    {
        "model_name":
            BEST_MODEL_NAME,

        "model":
            final_model,

        "reference_year":
            MODEL_REFERENCE_YEAR,

        "conformal_q_hat":
            CONFORMAL_Q_HAT,

        "model_features":
            MODEL_FEATURES,

        "categorical_features":
            CATEGORICAL_FEATURES,

        "manufacturer_canonical_map":
            MANUFACTURER_CANONICAL_MAP,

        "manufacturer_group_map":
            MANUFACTURER_GROUP_MAP,
    },
    MODEL_PATH,
)


print(
    "Saved:",
    MODEL_PATH
)

## 20. Reload smoke test

Never assume serialization worked just because no exception was raised.

This cell reloads the point model and confirms that it reproduces a prediction for the same engineered example.

In [ ]:
loaded_artifact = joblib.load(
    MODEL_PATH
)


loaded_model = loaded_artifact[
    "model"
]


reloaded_pred = predict_price_jpy(
    loaded_model,
    X_test.head(5),
)


original_pred = predict_price_jpy(
    final_model,
    X_test.head(5),
)


print(
    "Original:"
)

print(
    original_pred
)


print(
    "\nReloaded:"
)

print(
    reloaded_pred
)


assert np.allclose(
    original_pred,
    reloaded_pred
)


print(
    "\nReload test passed."
)

## Deployment gate

`Within_10pct` and `Within_20pct` are accuracy within a price tolerance; R² is not a percentage accuracy. The exported model is a snapshot model. Promote it only after checking the held-out metrics, interval coverage, segment errors, and a later-time holdout. Refit on fresh listings periodically.
